<h2 style="text-align: center;">Confecção do Mapa Interativo </h2>

<p style="text-align: center;">O mapa interativo realizado foi um mapa da cidade de Londrina separado por bairros. Esse mapa está disponível no site da prefeitura de Londrina no formato </p>

In [1]:
import json
import folium

bairrosList = []

with open("map_data\style.json", 'r') as file: 
    bairrosJSON = json.load(file)


In [2]:
import warnings
import pandas as pd

warnings.filterwarnings("ignore")

dadosBairros = pd.read_csv("csv files/dadosMapa.csv")

preco_por_bairro = dict(zip(dadosBairros["Bairro"], dadosBairros["Preço m²"]))
for feature in bairrosJSON["features"]:
    nome_bairro = feature["properties"]["BAIRRO"]
    preco = preco_por_bairro.get(nome_bairro)
    feature["properties"]["preco_m2"] = round(preco, 2) if preco is not None else "N/A"

In [3]:
from folium.features import DivIcon



m = folium.Map([-23.32243, -51.15817], zoom_start=12,tiles="cartodbpositron" )


ch = folium.Choropleth(
    geo_data = bairrosJSON,
    name="choropleth",
    data=dadosBairros,
    columns=["Bairro", "Preço m²"],
    key_on="feature.properties.BAIRRO",
    fill_color="OrRd",
    nan_fill_color="purple",
    nan_fill_opacity=0.4,
    fill_opacity=1,
    line_opacity=0.2,
    legend_name="Preço por m² (R$)",
    highlight=True,
    use_jenks=True
).add_to(m)


folium.GeoJsonTooltip(
    fields=["BAIRRO", "preco_m2"],
    aliases=["Bairro", "Preço médio m² (R$)"],
    localize=True,
    sticky=False,
    labels=True
).add_to(ch.geojson)

folium.LayerControl().add_to(m)

m.save("docs/mapaLondrina.html")
m